# Benford's Law (Model 3)

Does a set of numbers look *naturally occurring*
or does it look *fabricated/manipulated*?

- Benford's Law - in most real-world numerical datasets, the leading digit isn't uniformly distributed 1-9.
- It follows a specific curve where 1 shows up ~30% of the time and 9 shows up under 5% of the time.

- This test aims to catch human-manipulated numbers, which tend to distribute leading digits much more evenly.

In [1]:
import random
import math
from collections import Counter

# The expected Benford distribution for leading digits 1-9
BENFORD_EXPECTED = {d: math.log10(1 + 1/d) for d in range(1, 10)}

In [2]:
def leading_digit(n):
    """Extract the first significant digit of a number."""
    n = abs(n)
    while n >= 10:
        n /= 10
    while 0 < n < 1:
        n *= 10
    return int(n)

In [3]:
def benford_chi_square(numbers):
    """
    Compare the observed leading-digit distribution against Benford's
    expected distribution using a chi-square goodness-of-fit test.
    Returns the chi-square statistic and a per-digit breakdown.
    """
    digits = [leading_digit(n) for n in numbers if n != 0]
    counts = Counter(digits)
    total = len(digits)

    chi_sq = 0.0
    breakdown = []
    for d in range(1, 10):
        observed = counts.get(d, 0)
        expected = BENFORD_EXPECTED[d] * total
        chi_sq += (observed - expected) ** 2 / expected
        breakdown.append((d, observed, round(expected, 1)))

    return chi_sq, breakdown

In [4]:
def describe_result(chi_sq, n_digits):
    # Rough chi-square critical value for 8 degrees of freedom at p=0.05 is ~15.5
    critical_value = 15.51
    verdict = "LOOKS MANIPULATED (fails Benford's Law)" if chi_sq > critical_value else "looks naturally distributed"
    print(f"Chi-square statistic: {chi_sq:.2f}  (critical value at p=0.05: {critical_value})")
    print(f"n = {n_digits} numbers")
    print(f"Verdict: {verdict}\n")

In [5]:
def print_breakdown(breakdown):
    print(f"{'Digit':>5} {'Observed':>10} {'Expected':>10}")
    for d, obs, exp in breakdown:
        print(f"{d:>5} {obs:>10} {exp:>10}")
    print()

In [6]:
if __name__ == "__main__":
    random.seed(42)

    # --- Dataset 1: genuinely natural numbers (simulated invoice amounts).
    # Real financial data spans multiple orders of magnitude (a $12 invoice
    # and a $45,000 invoice both exist), and numbers spread log-uniformly
    # across orders of magnitude are exactly what produces a Benford-
    # compliant distribution --  standard simulation method ---
    natural_numbers = [round(10 ** random.uniform(0, 5), 2) for _ in range(500)]

    print("=== Dataset 1: simulated real invoice amounts ===")
    chi_sq, breakdown = benford_chi_square(natural_numbers)
    print_breakdown(breakdown)
    describe_result(chi_sq, len(natural_numbers))

    # --- Dataset 2: fabricated numbers (uniformly random -- human-manipulated
    # numbers tend to be presented with leading digits evenly spread) ---
    fabricated_numbers = [round(random.uniform(100, 999), 2) for _ in range(500)]

    print("=== Dataset 2: fabricated-looking numbers (uniform random) ===")
    chi_sq, breakdown = benford_chi_square(fabricated_numbers)
    print_breakdown(breakdown)
    describe_result(chi_sq, len(fabricated_numbers))

    # --- Dataset 3: too few numbers -- showing why sample size matters ---
    small_sample = natural_numbers[:8]
    print("=== Dataset 3: only 8 numbers (e.g. a handful of negotiation offers) ===")
    chi_sq, breakdown = benford_chi_square(small_sample)
    print_breakdown(breakdown)
    describe_result(chi_sq, len(small_sample))
    print("Notice: with only 8 data points, this test is close to meaningless --")
    print("this is the sample-size problem flagged earlier for Model 3.")

=== Dataset 1: simulated real invoice amounts ===
Digit   Observed   Expected
    1        167      150.5
    2         94       88.0
    3         66       62.5
    4         47       48.5
    5         35       39.6
    6         23       33.5
    7         25       29.0
    8         21       25.6
    9         22       22.9

Chi-square statistic: 7.66  (critical value at p=0.05: 15.51)
n = 500 numbers
Verdict: looks naturally distributed

=== Dataset 2: fabricated-looking numbers (uniform random) ===
Digit   Observed   Expected
    1         47      150.5
    2         51       88.0
    3         53       62.5
    4         49       48.5
    5         62       39.6
    6         58       33.5
    7         64       29.0
    8         59       25.6
    9         57       22.9

Chi-square statistic: 255.70  (critical value at p=0.05: 15.51)
n = 500 numbers
Verdict: LOOKS MANIPULATED (fails Benford's Law)

=== Dataset 3: only 8 numbers (like a handful of negotiation offers) ===
Digit 